# РГР з IoT - варіант 9
## Енергоефективний розумний будинок та обробка даних

Ноутбук запускає Python-конвеєр, який генерує 7 днів синтетичної телеметрії, очищує пропуски й аномалії, створює SQLite, виконує енергетичний аналіз, порівнює gzip/zlib/bzip2 та передає найкомпактніший набір до Firebase Realtime Database з подальшим зворотним читанням.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import shutil
from IPython.display import display, Image, FileLink

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_DIR = next((p.resolve() for p in candidates if (p / "code" / "smart_home_edge_pipeline.py").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Не знайдено code/smart_home_edge_pipeline.py. Відкрийте ноутбук із папки проєкту.")

SCRIPT = PROJECT_DIR / "code" / "smart_home_edge_pipeline.py"
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
print("Папка проєкту:", PROJECT_DIR)
print("Скрипт:", SCRIPT)
print("Результати:", OUTPUT_DIR)

## 1. Встановлення залежностей

In [ ]:
requirements = PROJECT_DIR / "requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)

## 2. Налаштування Firebase Realtime Database

In [ ]:
USE_FIREBASE = False  # Після налаштування Firebase змініть на True
FIREBASE_DATABASE_URL = "https://YOUR-DATABASE-NAME.firebaseio.com"  # Замініть точним URL із Firebase Console
FIREBASE_CREDENTIALS = PROJECT_DIR / "secrets" / "serviceAccountKey.json"

run_env = os.environ.copy()
run_env["OUTPUT_DIR"] = str(OUTPUT_DIR)
run_env["FIREBASE_ENABLED"] = "true" if USE_FIREBASE else "false"
if USE_FIREBASE:
    if "YOUR-DATABASE-NAME" in FIREBASE_DATABASE_URL or not FIREBASE_DATABASE_URL.startswith("https://"):
        raise ValueError("Вкажіть справжній URL Firebase Realtime Database.")
    if not FIREBASE_CREDENTIALS.is_file():
        raise FileNotFoundError(f"Не знайдено service-account JSON: {FIREBASE_CREDENTIALS}")
    run_env["FIREBASE_DATABASE_URL"] = FIREBASE_DATABASE_URL
    run_env["FIREBASE_CREDENTIALS"] = str(FIREBASE_CREDENTIALS.resolve())
else:
    run_env.pop("FIREBASE_DATABASE_URL", None)
    run_env.pop("FIREBASE_CREDENTIALS", None)
    print("Firebase вимкнено. Для повного виконання завдання увімкніть його після налаштування реквізитів.")

## 3. Запуск конвеєра

Скрипт створює сирі та очищені CSV, SQLite, статистику, графіки, стиснуті файли й `summary.json`. Якщо Firebase увімкнено, він завантажує bzip2-стиснений набір, читає його назад і перевіряє цілісність SHA-256.

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=str(SCRIPT.parent),
    env=run_env,
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"Конвеєр завершився з кодом {result.returncode}. Перевірте повідомлення вище та summary.json.")

verify_script = PROJECT_DIR / "code" / "verify_outputs.py"
verify_result = subprocess.run(
    [sys.executable, str(verify_script)],
    cwd=str(PROJECT_DIR),
    env=run_env,
    text=True,
    capture_output=True
)
print(verify_result.stdout)
if verify_result.stderr:
    print(verify_result.stderr)
if verify_result.returncode != 0:
    raise RuntimeError("Автоматична перевірка результатів не пройшла")


## 4. Перегляд результатів та перевірка Firebase

In [ ]:
import pandas as pd
summary = json.loads((OUTPUT_DIR / "summary.json").read_text(encoding="utf-8"))
daily = pd.read_csv(OUTPUT_DIR / "daily_summary.csv")
quality = pd.read_csv(OUTPUT_DIR / "quality_report.csv")
compression = pd.read_csv(OUTPUT_DIR / "compression_report.csv")

print("Варіант:", summary["variant"])
print("Кількість записів:", summary["rows"])
print("Споживання за 7 днів, кВт·год:", summary["total_energy_kwh_estimate"])
print("Економія освітлення в модельному сценарії, %:", summary["lighting_control_example"]["estimated_saving_percent"])
print("Firebase:", summary["firebase"]["status"])
display(daily)
display(quality)
display(compression)

if summary["firebase"]["status"] == "OK":
    downloaded = pd.read_csv(OUTPUT_DIR / "firebase_downloaded_readings.csv")
    print("Рядків, отриманих назад із Firebase:", len(downloaded))
    print("SHA-256 перевірено:", summary["firebase"].get("sha256_verified"))
elif USE_FIREBASE:
    raise RuntimeError("Firebase було ввімкнено, але зворотна перевірка не пройшла. Дивіться summary.json.")

In [ ]:
for filename in [
    "daily_energy.png", "temperature_timeseries.png", "illuminance_timeseries.png",
    "motion_by_hour.png", "compression_comparison.png"
]:
    image_path = OUTPUT_DIR / filename
    if image_path.is_file():
        print(filename)
        display(Image(filename=str(image_path)))

## 5. Експорт результатів

In [ ]:
archive_path = PROJECT_DIR / "RGR_IoT_variant9_results"
out_zip = shutil.make_archive(str(archive_path), "zip", root_dir=str(OUTPUT_DIR))
print("Архів результатів:", out_zip)
display(FileLink(out_zip))